In [45]:
from kafka import KafkaConsumer

server = 'localhost:9092'
topic_name = 'rides'

In [37]:
from models import Ride, ride_deserializer

In [46]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='rides-database',
    value_deserializer=ride_deserializer
)

In [20]:
import sys
!pip install psycopg2-binary


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /usr/local/python/3.12.1/bin/python3 -m pip install --upgrade pip


In [23]:
import sys
print(sys.executable)

/workspaces/streaming-workshop/.venv/bin/python


In [24]:
!which python
!python -m pip show psycopg2-binary

/workspaces/streaming-workshop/.venv/bin/python
/workspaces/streaming-workshop/.venv/bin/python: No module named pip


In [25]:
import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install psycopg2-binary

Looking in links: /tmp/tmp0ffc8yx0
Processing /tmp/tmp0ffc8yx0/pip-23.2.1-py3-none-any.whl
  Obtaining dependency information for psycopg2-binary from https://files.pythonhosted.org/packages/30/da/4e42788fb811bbbfd7b7f045570c062f49e350e1d1f3df056c3fb5763353/psycopg2_binary-2.9.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 15.8 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip


In [39]:
import psycopg2
print("psycopg2 installed")

psycopg2 installed


In [43]:
# import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5433,
    database='postgres',
    user='postgres',
    password='postgres'
)

conn.autocommit = True
cur = conn.cursor()

In [ ]:
from datetime import datetime

print(f"Listening to {topic_name} and writing to PostgreSQL...")

count = 0
for message in consumer:
    ride = message.value
    pickup_dt = datetime.fromtimestamp(ride.tpep_pickup_datetime / 1000)
    cur.execute(
        """INSERT INTO processed_events
           (PULocationID, DOLocationID, trip_distance, total_amount, pickup_datetime)
           VALUES (%s, %s, %s, %s, %s)""",
        (ride.PULocationID, ride.DOLocationID,
         ride.trip_distance, ride.total_amount, pickup_dt)
    )
    count += 1
    if count % 100 == 0:
        print(f"Inserted {count} rows...")

consumer.close()
cur.close()
conn.close()

Listening to rides and writing to PostgreSQL...
Inserted 100 rows...
Inserted 200 rows...
Inserted 300 rows...
Inserted 400 rows...
Inserted 500 rows...
Inserted 600 rows...
Inserted 700 rows...
Inserted 800 rows...
Inserted 900 rows...
Inserted 1000 rows...
Inserted 1100 rows...
Inserted 1200 rows...
Inserted 1300 rows...
Inserted 1400 rows...
Inserted 1500 rows...
